# 01 - Simulation EDA

Exploratory analysis of the synthetic credit-card portfolio. This notebook validates the generated users, visualizes the main distributions, and checks whether the simulator calibration looks internally consistent before policy training.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.simulator import simulate_month


## Section 1: Load and Validate Synthetic Users

We start by loading the generated user portfolio and checking the basics: schema shape, data types, nulls, categorical balance, and a simulator calibration replay to ensure the month-one aggregate default rate still falls in the expected range.

In [2]:
users_path = Path('../data/synthetic_users.csv')
if not users_path.exists():
    raise FileNotFoundError('Run src/simulator.py first to generate data/synthetic_users.csv')

users_df = pd.read_csv(users_path)
print('Shape:', users_df.shape)
display(users_df.head())
display(users_df.dtypes.rename('dtype').to_frame())

null_summary = users_df.isnull().sum().rename('null_count').to_frame()
display(null_summary)

categorical_columns = ['age_bucket', 'income_bucket', 'employment_type', 'spending_category', 'risk_tier']
for column in categorical_columns:
    print(f'Value counts for {column}:')
    display(users_df[column].value_counts(dropna=False).rename('count').to_frame())

month_one_df = simulate_month(users_df, month_num=1, economic_stress=1.0)
aggregate_default_rate = month_one_df['did_default'].mean()
print(f'Aggregate month-1 default rate: {aggregate_default_rate:.4%}')

assert 0.025 <= aggregate_default_rate <= 0.035, 'Aggregate default rate is outside the 2.5%-3.5% target band.'
assert users_df['risk_tier'].nunique() == 4, 'Expected all four risk tiers to be present.'
assert not users_df['user_id'].duplicated().any(), 'Duplicate user_id values found.'
print('Validation checks passed.')


Shape: (10000, 13)


,user_id,age_bucket,income_bucket,cibil_score,employment_type,payment_streak,utilization_ratio,spending_category,account_age_months,delinquency_count,transaction_frequency,risk_tier,initial_credit_limit
0,USER_00001,36-50,low,608,gig,4,0.7418,essentials,24,3,3,Subprime,32355.79
1,USER_00002,18-25,low,731,gig,22,0.3826,essentials,80,5,13,Near-Prime,74087.30
2,USER_00003,51+,high,615,salaried,7,0.5026,mixed,40,4,23,Subprime,47887.94
3,USER_00004,26-35,low,676,salaried,19,0.2156,mixed,11,0,25,Near-Prime,71719.32
4,USER_00005,36-50,very_high,841,salaried,23,0.4313,lifestyle,19,1,26,Prime,500000.00


,dtype
user_id,str
age_bucket,str
income_bucket,str
cibil_score,int64
employment_type,str
payment_streak,int64
utilization_ratio,float64
spending_category,str
account_age_months,int64
delinquency_count,int64


,null_count
user_id,0
age_bucket,0
income_bucket,0
cibil_score,0
employment_type,0
payment_streak,0
utilization_ratio,0
spending_category,0
account_age_months,0
delinquency_count,0


Value counts for age_bucket:


...

,count
spending_category,
mixed,4158
essentials,3687
lifestyle,2155


Value counts for risk_tier:


,count
risk_tier,
Prime,4034
Near-Prime,3024
Subprime,1957
Deep-Subprime,985


Aggregate month-1 default rate: 3.4600%


Validation checks passed.


## Section 2: Distribution Plots

These views show how credit quality and limits vary across risk tiers, and how utilization interacts with repayment behavior. They help confirm that the synthetic data separates risk cohorts in a believable way.

In [3]:
fig_cibil = px.histogram(
    users_df,
    x='cibil_score',
    color='risk_tier',
    opacity=0.55,
    barmode='overlay',
    nbins=40,
    title='CIBIL Score Distribution by Risk Tier',
)
fig_cibil.update_layout(xaxis_title='CIBIL Score', yaxis_title='User Count')
fig_cibil.show()

fig_limit = px.box(
    users_df,
    x='risk_tier',
    y='initial_credit_limit',
    color='risk_tier',
    title='Initial Credit Limit by Risk Tier',
)
fig_limit.update_layout(xaxis_title='Risk Tier', yaxis_title='Initial Credit Limit (INR)', showlegend=False)
fig_limit.show()

scatter_sample = users_df.sample(min(len(users_df), 2500), random_state=42)
fig_util = px.scatter(
    scatter_sample,
    x='utilization_ratio',
    y='payment_streak',
    color='risk_tier',
    opacity=0.7,
    title='Utilization Ratio vs Payment Streak by Risk Tier',
)
fig_util.update_layout(xaxis_title='Utilization Ratio', yaxis_title='Payment Streak (months)')
fig_util.show()


## Section 3: Correlation Analysis

A correlation heatmap gives a fast read on the strongest numeric relationships in the synthetic data. We also isolate the top drivers of delinquency to sanity-check whether the portfolio risk mechanics line up with expectations.

In [4]:
numeric_columns = [
    'cibil_score',
    'payment_streak',
    'utilization_ratio',
    'account_age_months',
    'delinquency_count',
    'transaction_frequency',
    'initial_credit_limit',
]
corr_df = users_df[numeric_columns].corr(numeric_only=True)

fig_corr = px.imshow(
    corr_df,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Numeric Feature Correlation Heatmap',
)
fig_corr.show()

top_delinquency_features = (
    corr_df['delinquency_count']
    .drop(labels='delinquency_count')
    .abs()
    .sort_values(ascending=False)
    .head(3)
    .rename('abs_correlation_with_delinquency_count')
)
print('Top 3 features correlated with delinquency_count:')
display(top_delinquency_features.to_frame())


Top 3 features correlated with delinquency_count:


,abs_correlation_with_delinquency_count
cibil_score,0.556771
payment_streak,0.476490
initial_credit_limit,0.462115


## Section 4: Cohort Summary Table

This summary collapses the synthetic user base to the risk-tier level so we can quickly compare average credit score, average starting limit, average delinquency load, and cohort size.

In [5]:
cohort_summary = (
    users_df.groupby('risk_tier', as_index=False)
    .agg(
        mean_cibil=('cibil_score', 'mean'),
        mean_limit=('initial_credit_limit', 'mean'),
        mean_delinquency=('delinquency_count', 'mean'),
        user_count=('user_id', 'count'),
    )
    .sort_values('mean_limit', ascending=False)
)
display(cohort_summary.style.format({'mean_cibil': '{:.1f}', 'mean_limit': '{:,.2f}', 'mean_delinquency': '{:.2f}'}))


,risk_tier,mean_cibil,mean_limit,mean_delinquency,user_count
2,Prime,829.9,"327,976.59",0.16,4034
1,Near-Prime,714.1,"98,384.27",0.67,3024
3,Subprime,622.6,"42,240.50",1.76,1957
0,Deep-Subprime,432.1,"20,793.99",2.62,985
